[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/certified-journeys/certified-journeys.github.io/blob/main/courses/prompt-engineering-certified/notebooks/day-04-iterative-refinement.ipynb#scrollTo=a1b2c3d4)

---
# Day 4 · Iterative Prompt Refinement
**certified-journeys / prompt-engineering-certified** · Day 4 · Practice

> **Goal for today:** Diagnose why a prompt fails, build a structured refinement log, and iterate systematically until your summarization prompt passes 8 out of 10 test cases.


In [ ]:
%pip install -q openai


## Step 1 · Setting up the mock LLM client

All code in this notebook uses a **mock** OpenAI client so it runs in Colab without a real API key. In production, replace `MockOpenAI` with `openai.OpenAI(api_key=...)` and `gpt-4o-mini` as the model.

| Mock behaviour | Production equivalent |
|---|---|
| Deterministic canned responses | Real LLM completions via `client.chat.completions.create` |
| `MockCompletion` object | `openai.types.chat.ChatCompletion` |
| No network call | HTTPS to `api.openai.com` |


In [ ]:
# ── Mock OpenAI client ──────────────────────────────────────────
# In production: from openai import OpenAI; client = OpenAI(api_key="sk-...")

from dataclasses import dataclass, field
from typing import List, Optional
import textwrap, random

@dataclass
class MockMessage:
    content: str
    role: str = "assistant"

@dataclass
class MockChoice:
    message: MockMessage
    index: int = 0
    finish_reason: str = "stop"

@dataclass
class MockCompletion:
    choices: List[MockChoice]
    model: str = "gpt-4o-mini"

# Registry of canned responses keyed by prompt snippet
_RESPONSES = {}

def register_response(key: str, response: str):
    """Register a canned response for a keyword pattern."""
    _RESPONSES[key] = response

class _Chat:
    class _Completions:
        def create(self, model: str, messages: list, **kwargs) -> MockCompletion:
            content = messages[-1]["content"] if messages else ""
            # Match registered canned response
            for key, response in _RESPONSES.items():
                if key.lower() in content.lower():
                    return MockCompletion([MockChoice(MockMessage(response))])
            # Default fallback
            return MockCompletion([MockChoice(MockMessage("[mock response]"))])
    completions = _Completions()

class MockOpenAI:
    chat = _Chat()

client = MockOpenAI()
MODEL = "gpt-4o-mini"
print("Mock client ready — replace with openai.OpenAI() for live calls")


### What just happened?
- **`MockOpenAI`** mirrors the real `openai.OpenAI` interface so all downstream code is identical.
- **`register_response`** lets us inject deterministic outputs keyed by substring match — essential for reproducible testing.
- Swapping to production requires one line: `client = openai.OpenAI(api_key=os.environ["OPENAI_API_KEY"])`.


## Step 2 · Defining a test set for summarization

A **test set** lets you measure prompt quality objectively. Each example has:
- `input`: the text to summarize
- `expected_keywords`: words that must appear in a good summary
- `forbidden_keywords`: words that signal hallucination or off-topic output
- `max_words`: length constraint

We'll run 10 test cases and track pass/fail rates across prompt versions.


In [ ]:
# ── Test set: 10 summarization examples ────────────────────────

TEST_SET = [
    {
        "id": 1,
        "input": "The Apollo 11 mission landed astronauts Neil Armstrong and Buzz Aldrin on the Moon on July 20, 1969. Armstrong became the first human to walk on the lunar surface. The mission returned 47.5 pounds of lunar material to Earth.",
        "expected_keywords": ["apollo", "moon", "armstrong"],
        "forbidden_keywords": ["mars", "saturn"],
        "max_words": 30,
    },
    {
        "id": 2,
        "input": "Python was created by Guido van Rossum and first released in 1991. It emphasizes code readability and uses significant whitespace. Python 3 was released in 2008 and is not backward compatible with Python 2.",
        "expected_keywords": ["python", "guido", "readability"],
        "forbidden_keywords": ["java", "ruby"],
        "max_words": 30,
    },
    {
        "id": 3,
        "input": "Climate change refers to long-term shifts in global temperatures and weather patterns. While some changes are natural, human activities—especially burning fossil fuels—have been the primary driver since the 1800s.",
        "expected_keywords": ["climate", "temperature", "fossil"],
        "forbidden_keywords": ["nuclear", "asteroid"],
        "max_words": 30,
    },
    {
        "id": 4,
        "input": "The Great Wall of China is a series of fortifications built across the northern borders of China. Construction began as early as the 7th century BC. The wall stretches approximately 13,170 miles.",
        "expected_keywords": ["wall", "china", "fortif"],
        "forbidden_keywords": ["rome", "egypt"],
        "max_words": 30,
    },
    {
        "id": 5,
        "input": "Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce oxygen and energy in the form of sugar. It occurs mainly in the chloroplasts of plant cells.",
        "expected_keywords": ["photosynthesis", "sunlight", "oxygen"],
        "forbidden_keywords": ["animal", "respiration"],
        "max_words": 30,
    },
    {
        "id": 6,
        "input": "Machine learning is a branch of artificial intelligence that allows systems to learn and improve from experience without being explicitly programmed. It focuses on developing programs that access data and learn from it.",
        "expected_keywords": ["machine learning", "data", "learn"],
        "forbidden_keywords": ["robot", "hardware"],
        "max_words": 30,
    },
    {
        "id": 7,
        "input": "Shakespeare wrote 37 plays and 154 sonnets. His works have been translated into every major language and are performed more often than those of any other playwright. He was born in Stratford-upon-Avon in 1564.",
        "expected_keywords": ["shakespeare", "plays", "sonnets"],
        "forbidden_keywords": ["dickens", "novel"],
        "max_words": 30,
    },
    {
        "id": 8,
        "input": "The internet was developed by ARPA in the late 1960s as ARPANET, a network connecting US universities. Tim Berners-Lee invented the World Wide Web in 1989, making the internet widely accessible.",
        "expected_keywords": ["internet", "arpanet", "web"],
        "forbidden_keywords": ["telephone", "radio"],
        "max_words": 30,
    },
    {
        "id": 9,
        "input": "DNA, or deoxyribonucleic acid, carries the genetic instructions for the development, functioning, and reproduction of all living organisms. James Watson and Francis Crick described its double-helix structure in 1953.",
        "expected_keywords": ["dna", "genetic", "watson"],
        "forbidden_keywords": ["protein", "rna"],
        "max_words": 30,
    },
    {
        "id": 10,
        "input": "The Renaissance was a cultural and intellectual movement that began in Italy in the 14th century and spread across Europe. It marked the transition from the Middle Ages to modernity and led to advances in art, science, and philosophy.",
        "expected_keywords": ["renaissance", "italy", "art"],
        "forbidden_keywords": ["industrial", "revolution"],
        "max_words": 30,
    },
]

print(f"Test set loaded: {len(TEST_SET)} examples")


### What just happened?
- **10 test cases** span diverse domains so we can't overfit a prompt to one topic.
- **`expected_keywords`** catch content omissions — the most common failure mode.
- **`forbidden_keywords`** catch hallucinations — content the model invented.
- **`max_words`** enforces the format constraint — a summary that's 200 words isn't a summary.


## Step 3 · The evaluator and refinement log

Before iterating, we need an **automated evaluator** that scores each response. This lets us compare prompt versions objectively.

Failure mode taxonomy:
| Failure mode | Symptom | Fix strategy |
|---|---|---|
| Wrong format | Too long, bullet points instead of prose | Add explicit format constraint |
| Hallucination | Forbidden keyword found | Add "do not add information" instruction |
| Off-topic | Missing expected keywords | Add "focus on" instruction |
| Too vague | No specific facts | Add "include key facts" instruction |


In [ ]:
# ── Evaluator ───────────────────────────────────────────────────

def evaluate_summary(summary: str, test_case: dict) -> dict:
    """Score a summary against test case criteria. Returns a result dict."""
    words = summary.split()
    word_count = len(words)
    summary_lower = summary.lower()

    # Check each criterion independently
    length_ok = word_count <= test_case["max_words"]
    keywords_found = [kw for kw in test_case["expected_keywords"] if kw in summary_lower]
    content_ok = len(keywords_found) >= len(test_case["expected_keywords"]) * 0.67  # 2/3 threshold
    hallucinations = [kw for kw in test_case["forbidden_keywords"] if kw in summary_lower]
    no_hallucination = len(hallucinations) == 0

    passed = length_ok and content_ok and no_hallucination

    # Identify primary failure mode for the refinement log
    failure_mode = None
    if not length_ok:
        failure_mode = f"too_long ({word_count} words, max {test_case['max_words']})"
    elif not no_hallucination:
        failure_mode = f"hallucination ({hallucinations})"
    elif not content_ok:
        failure_mode = f"off_topic (missing {[kw for kw in test_case['expected_keywords'] if kw not in summary_lower]})"

    return {
        "id": test_case["id"],
        "passed": passed,
        "word_count": word_count,
        "failure_mode": failure_mode,
        "summary_preview": summary[:80] + "..." if len(summary) > 80 else summary,
    }


def run_prompt_version(prompt_template: str, version_name: str) -> dict:
    """Run a prompt version against all 10 test cases. Return aggregated results."""
    results = []
    for case in TEST_SET:
        prompt = prompt_template.format(text=case["input"])
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}]
        )
        summary = response.choices[0].message.content
        result = evaluate_summary(summary, case)
        results.append(result)

    passed_count = sum(1 for r in results if r["passed"])
    return {
        "version": version_name,
        "passed": passed_count,
        "total": len(TEST_SET),
        "pass_rate": f"{passed_count}/{len(TEST_SET)}",
        "results": results,
    }


print("Evaluator ready")


### What just happened?
- **`evaluate_summary`** is a rule-based scorer — fast, deterministic, no LLM needed for basic criteria.
- The **failure mode** field is the key output: it tells us which single thing to fix next.
- **`run_prompt_version`** wraps the full eval loop so we can compare versions in one function call.


## Step 4 · Version 1 — The naive prompt (baseline)

We start with the simplest possible prompt to establish a baseline. Expect it to fail on format and sometimes content.


In [ ]:
# ── Version 1: Naive prompt ─────────────────────────────────────

PROMPT_V1 = "Summarize this text:\n\n{text}"

# Register mock responses that simulate a naive model: often too long, sometimes hallucinating
register_response("apollo 11",
    "The Apollo 11 mission in 1969 landed humans on the moon. Neil Armstrong and Buzz Aldrin walked on the lunar surface. Armstrong said 'one small step for man.' They also visited Mars later. This was a great achievement for humanity and NASA and the United States space program which had been competing with the Soviet Union.")
register_response("python was created",
    "Python is a programming language created by Guido van Rossum in 1991. It emphasizes readability. Python 3 came in 2008.")
register_response("climate change",
    "Climate change involves shifts in global temperatures. Human activities and nuclear energy are causing it, especially since the industrial revolution began with factories.")
register_response("great wall",
    "The Great Wall of China is a fortification built along China's northern border starting in the 7th century BC, stretching 13,170 miles.")
register_response("photosynthesis",
    "Photosynthesis is how plants use sunlight, water, and CO2 to produce oxygen and sugar, occurring in chloroplasts. Plants also perform animal respiration at night which is the opposite process used to release energy.")
register_response("machine learning is a branch",
    "Machine learning is a part of AI that lets systems learn and improve from data without explicit programming. It is used in many hardware applications.")
register_response("shakespeare wrote",
    "Shakespeare wrote 37 plays and 154 sonnets, translated into every major language. He was born in Stratford-upon-Avon in 1564.")
register_response("arpanet",
    "The internet was developed by ARPA in the 1960s as ARPANET, connecting US universities. Tim Berners-Lee invented the World Wide Web in 1989, similar to how the telephone changed communication.")
register_response("deoxyribonucleic acid",
    "DNA carries genetic instructions for all living organisms. Watson and Crick described its double-helix structure in 1953. Proteins and RNA are closely related.")
register_response("renaissance was a cultural",
    "The Renaissance was a cultural and intellectual movement beginning in Italy in the 14th century, spreading across Europe and transitioning society from the Middle Ages to modernity with advances in art, science, philosophy. It eventually led to the industrial revolution that transformed manufacturing across the continent.")

v1_results = run_prompt_version(PROMPT_V1, "v1_naive")
print(f"\nVersion 1 (naive): {v1_results['pass_rate']} passed")
print("\nFailure modes:")
for r in v1_results["results"]:
    status = "PASS" if r["passed"] else "FAIL"
    mode = r["failure_mode"] or "—"
    print(f"  [{status}] Case {r['id']:2d}: {mode}")


### What just happened?
- The **naive prompt** gives the model no constraints on length, content focus, or accuracy.
- We can see **hallucination** (invented facts), **too long** responses, and **off-topic** content.
- The baseline score lets us measure whether each fix actually helps.


## Step 5 · Refinement iterations and the 3-column log

We fix **one variable at a time** — the most important discipline in prompt engineering. The refinement log captures exactly what changed and why.

| Prompt version | Failure observed | Fix applied |
|---|---|---|
| v1_naive | Too long; hallucinations | Add word limit + "only use information in the text" |
| v2_constrained | Still some off-topic content | Add explicit "do not add information not in the text" |
| v3_focused | Occasional missing keywords | Add "include key facts: who, what, when" instruction |


In [ ]:
# ── Re-register responses that improve with better prompts ──────
# Simulate a better model response when the prompt includes constraints

def reset_responses_for_version(version: int):
    """Reset the mock registry with responses appropriate for each prompt version."""
    global _RESPONSES
    _RESPONSES = {}

    v1_responses = [
        ("apollo 11", "The Apollo 11 mission in 1969 landed humans on the moon. Neil Armstrong and Buzz Aldrin walked on the lunar surface. Armstrong said 'one small step for man.' They also visited Mars later. This was a great achievement for humanity and the United States space program."),
        ("python was created", "Python is a programming language created by Guido van Rossum in 1991. It emphasizes readability. Python 3 came in 2008."),
        ("climate change", "Climate change involves shifts in global temperatures. Nuclear energy and human activities are causing it."),
        ("great wall", "The Great Wall of China is a fortification along China's northern border starting in the 7th century BC."),
        ("photosynthesis", "Photosynthesis is how plants use sunlight to produce oxygen and sugar in chloroplasts. Animal respiration is the reverse process."),
        ("machine learning is a branch", "Machine learning is a part of AI that lets systems learn from data. It powers many hardware devices."),
        ("shakespeare wrote", "Shakespeare wrote 37 plays and 154 sonnets and was born in Stratford-upon-Avon in 1564."),
        ("arpanet", "The internet started as ARPANET in the 1960s. Tim Berners-Lee invented the World Wide Web, like how the telephone changed communication."),
        ("deoxyribonucleic acid", "DNA carries genetic instructions for all living organisms and consists of proteins and RNA. Watson and Crick described it in 1953."),
        ("renaissance was a cultural", "The Renaissance was a cultural movement starting in Italy in the 14th century that led to the industrial revolution and advances in art and science."),
    ]
    v2_responses = [
        ("apollo 11", "Apollo 11 landed astronauts Armstrong and Aldrin on the Moon in July 1969, returning 47.5 pounds of lunar material."),
        ("python was created", "Python, created by Guido van Rossum in 1991, emphasizes readability and uses significant whitespace; Python 3 launched in 2008."),
        ("climate change", "Climate change involves long-term shifts in global temperatures caused primarily by human burning of fossil fuels since the 1800s."),
        ("great wall", "The Great Wall of China, a series of fortifications along its northern border, began construction in the 7th century BC and stretches 13,170 miles."),
        ("photosynthesis", "Photosynthesis is the process plants use to convert sunlight, water, and CO2 into oxygen and sugar, occurring in chloroplasts."),
        ("machine learning is a branch", "Machine learning, a branch of AI, enables systems to learn from data and improve without explicit programming."),
        ("shakespeare wrote", "Shakespeare wrote 37 plays and 154 sonnets, performed more than any other playwright; he was born in Stratford-upon-Avon in 1564."),
        ("arpanet", "The internet originated as ARPANET in the 1960s; Tim Berners-Lee invented the World Wide Web in 1989."),
        ("deoxyribonucleic acid", "DNA carries genetic instructions for all living organisms; Watson and Crick described its double-helix structure in 1953."),
        ("renaissance was a cultural", "The Renaissance was a cultural and intellectual movement beginning in 14th-century Italy that spread across Europe, advancing art, science, and philosophy."),
    ]
    v3_responses = [
        ("apollo 11", "Apollo 11 (July 1969) landed astronauts Neil Armstrong and Buzz Aldrin on the Moon, returning 47.5 pounds of lunar material to Earth."),
        ("python was created", "Guido van Rossum created Python in 1991, emphasizing code readability and significant whitespace; Python 3, released in 2008, is not backward compatible."),
        ("climate change", "Climate change refers to long-term shifts in global temperatures, driven primarily by human fossil fuel burning since the 1800s."),
        ("great wall", "The Great Wall of China, a series of northern border fortifications, was built from the 7th century BC and stretches 13,170 miles."),
        ("photosynthesis", "Photosynthesis allows plants to convert sunlight, water, and CO2 into oxygen and sugar inside chloroplasts."),
        ("machine learning is a branch", "Machine learning, a branch of AI, enables systems to learn from experience and data without explicit programming."),
        ("shakespeare wrote", "Shakespeare wrote 37 plays and 154 sonnets, born in Stratford-upon-Avon in 1564, and is the most performed playwright."),
        ("arpanet", "The internet began as ARPANET (1960s) connecting US universities; Tim Berners-Lee invented the World Wide Web in 1989."),
        ("deoxyribonucleic acid", "DNA carries genetic instructions for all living organisms; Watson and Crick described its double-helix structure in 1953."),
        ("renaissance was a cultural", "The Renaissance was a 14th-century Italian cultural movement that spread across Europe, advancing art, science, and philosophy."),
    ]

    responses = {1: v1_responses, 2: v2_responses, 3: v3_responses}.get(version, v3_responses)
    for key, val in responses:
        _RESPONSES[key] = val

# ── Version 2: Add word limit + no-hallucination instruction ────
reset_responses_for_version(2)

PROMPT_V2 = (
    "Summarize the following text in 25 words or fewer. "
    "Only use information from the text — do not add any facts not present in the passage.\n\n"
    "{text}"
)

v2_results = run_prompt_version(PROMPT_V2, "v2_constrained")
print(f"Version 2 (constrained): {v2_results['pass_rate']} passed")
for r in v2_results["results"]:
    status = "PASS" if r["passed"] else "FAIL"
    mode = r["failure_mode"] or "—"
    print(f"  [{status}] Case {r['id']:2d}: {mode}")


### What just happened?
- **Fix 1 (word limit):** Eliminates the `too_long` failure mode entirely.
- **Fix 2 (no-hallucination instruction):** Reduces invented facts by anchoring the model to the source.
- We changed **two things at once** here deliberately to show a common mistake — ideally isolate each.


In [ ]:
# ── Version 3: Add key-facts instruction ───────────────────────
reset_responses_for_version(3)

PROMPT_V3 = (
    "Summarize the following text in 25 words or fewer. "
    "Include the key facts (who, what, when) from the passage. "
    "Do not add any information not present in the source text.\n\n"
    "{text}"
)

v3_results = run_prompt_version(PROMPT_V3, "v3_focused")
print(f"Version 3 (focused):     {v3_results['pass_rate']} passed")
for r in v3_results["results"]:
    status = "PASS" if r["passed"] else "FAIL"
    mode = r["failure_mode"] or "—"
    print(f"  [{status}] Case {r['id']:2d}: {mode}")

# ── Summary comparison table ────────────────────────────────────
print("\n" + "="*50)
print("Refinement log summary:")
print(f"{'Version':<20} {'Pass rate':<12} {'Fix applied'}")
print("-"*60)
log = [
    ("v1_naive",       v1_results['pass_rate'], "Baseline — no constraints"),
    ("v2_constrained", v2_results['pass_rate'], "Added word limit + no-hallucination"),
    ("v3_focused",     v3_results['pass_rate'], "Added 'include key facts: who/what/when'"),
]
for version, rate, fix in log:
    print(f"{version:<20} {rate:<12} {fix}")


### What just happened?
- **Fix 3 (key facts):** Fills in missing expected keywords by directing the model to surface who, what, and when.
- The **refinement log** gives a clear audit trail — if a fix regresses performance, we can roll back.
- **v3 hits 10/10** in this simulation; in practice aim for 8/10 as the target gate.


## Step 6 · Post-mortem: what made the original prompt fail?

A post-mortem documents the root cause and the minimal fix — essential for a team working on the same prompt system.


In [ ]:
# ── Post-mortem report ──────────────────────────────────────────

def generate_postmortem(v1: dict, vfinal: dict, fixes: list) -> str:
    """Generate a structured post-mortem comparing baseline to final prompt."""
    v1_fails = [r for r in v1["results"] if not r["passed"]]
    vf_fails = [r for r in vfinal["results"] if not r["passed"]]

    failure_modes_v1 = {}
    for r in v1_fails:
        mode = r["failure_mode"].split(" (")[0] if r["failure_mode"] else "unknown"
        failure_modes_v1[mode] = failure_modes_v1.get(mode, 0) + 1

    lines = [
        "=" * 60,
        "POST-MORTEM: Summarization Prompt Refinement",
        "=" * 60,
        f"Baseline (v1) pass rate : {v1['pass_rate']}",
        f"Final (v3) pass rate    : {vfinal['pass_rate']}",
        "",
        "── Root cause analysis ──",
    ]
    for mode, count in failure_modes_v1.items():
        lines.append(f"  • {mode}: {count} case(s)")

    lines += [
        "",
        "── Fixes applied (in order) ──",
    ]
    for i, fix in enumerate(fixes, 1):
        lines.append(f"  {i}. {fix}")

    lines += [
        "",
        "── Minimal fix ──",
        "  The original prompt provided no output constraints.",
        "  The single most impactful fix was adding a word limit (25 words),",
        "  which eliminated the format failure mode entirely.",
        "  The hallucination and off-topic fixes required explicit instructions",
        "  but together accounted for all remaining failures.",
        "",
        "── Lessons ──",
        "  1. Models default to verbose output — always specify a word/sentence limit.",
        "  2. 'Do not hallucinate' alone is insufficient — anchor to source text explicitly.",
        "  3. Specifying what to include (key facts) is more reliable than relying on defaults.",
        "  4. Fix one variable per iteration to isolate cause and effect.",
        "=" * 60,
    ]

    return "\n".join(lines)

fixes_applied = [
    "Added 25-word limit (v1→v2): eliminated 'too_long' failures",
    "Added no-hallucination instruction (v1→v2): eliminated invented facts",
    "Added 'include key facts: who/what/when' (v2→v3): eliminated off-topic failures",
]

postmortem = generate_postmortem(v1_results, v3_results, fixes_applied)
print(postmortem)


### What just happened?
- The **post-mortem** codifies what we learned so the next engineer doesn't repeat the same iteration.
- **Root cause was structural**: no output constraints → model chose its own defaults, which were wrong.
- The **minimal fix** was the word limit — everything else was refinement on top of a constrained baseline.
- Always document failure modes **before** writing fixes, not after.


In [ ]:
# Challenge: Iterative refinement on a classification prompt
# ─────────────────────────────────────────────────────────────
# The prompt below misclassifies sentiment. Your task:
#   1. Run the baseline and observe the failure mode
#   2. Apply ONE fix at a time — log the version and what changed
#   3. Reach 8/10 pass rate on the classification test set
#   4. Write a one-paragraph post-mortem at the end
#
# Starter baseline prompt — deliberately underspecified:
CLASSIFICATION_PROMPT_V1 = "Classify the sentiment of this text:\n\n{text}"

CLASSIFICATION_TEST = [
    {"id": 1, "input": "I loved this product, it exceeded my expectations!", "expected": "positive"},
    {"id": 2, "input": "Terrible experience. Never buying again.", "expected": "negative"},
    {"id": 3, "input": "It was okay, nothing special.", "expected": "neutral"},
    {"id": 4, "input": "The quality is amazing but the delivery was slow.", "expected": "mixed"},
    {"id": 5, "input": "Absolutely perfect in every way!", "expected": "positive"},
    {"id": 6, "input": "Not what I expected. Disappointed.", "expected": "negative"},
    {"id": 7, "input": "Does the job. Nothing more.", "expected": "neutral"},
    {"id": 8, "input": "Great features but crashes constantly.", "expected": "mixed"},
    {"id": 9, "input": "Five stars! Highly recommend!", "expected": "positive"},
    {"id": 10, "input": "Waste of money.", "expected": "negative"},
]

# TODO: Fill in your implementation
# Step 1: Write an evaluator for classification (hint: check if expected label appears in output)
def evaluate_classification(output: str, test_case: dict) -> dict:
    # Your code here
    pass

# Step 2: Run CLASSIFICATION_PROMPT_V1 against CLASSIFICATION_TEST
# Step 3: Build refinement log — improve the prompt in at least 3 iterations
# CLASSIFICATION_PROMPT_V2 = "..."  # Fix 1: ?
# CLASSIFICATION_PROMPT_V3 = "..."  # Fix 2: ?
# Step 4: Print your 3-column refinement log (version | failure observed | fix applied)
# Step 5: Write a post-mortem paragraph explaining the root cause and minimal fix


---
## Day 4 key concepts recap
| Concept | What to remember |
|---|---|
| Fix one variable at a time | Changing two things at once makes it impossible to know which worked |
| Failure mode taxonomy | Too long / hallucination / off-topic — diagnose before fixing |
| Refinement log | Three columns: prompt version, failure observed, fix applied |
| Word limit as first fix | Models default to verbose — always constrain length first |
| Post-mortem | Documents root cause + minimal fix so the team doesn't re-do the work |
| Pass rate gate | 8/10 is a practical threshold for moving to the next phase |

> **Tip:** Fix one variable at a time. Treat prompt refinement like an A/B test — isolate each change.

---
## What's next
**Day 5** → Structured Outputs — JSON Mode and Function Calling: get exact-shape output from an LLM using JSON mode and Pydantic validation.

Mark Day 4 complete in your [tracker](../index.html).
